In [2]:
import duckdb
import pandas as pd

con = duckdb.connect(
    "../data/processed/chess.duckdb"
)

features = con.sql("""
SELECT *
FROM features
""").df()

features.head()

,game_id,date,white_id,black_id,elo_white,elo_black,elo_diff,elo_mean,white_games_count,black_games_count,white_form_10,black_form_10,white_form_30,black_form_30,days_since_last_game_white,days_since_last_game_black,h2h_games,h2h_white_score,y
0,80d9ebb6427d4b6afc3efc090fa9d0a1,2023-04-17,8618399,63100630,1791.0,1619.0,172.0,1705.0,0,0,0.5,0.5,0.5,0.5,-1.0,-1.0,0,0.5,2
1,07324326ce0e6ec7453ecc12163703ea,2023-04-17,8642214,8642419,1669.0,1522.0,147.0,1595.5,0,0,0.5,0.5,0.5,0.5,-1.0,-1.0,0,0.5,2
2,66cc79839f5bdb967949a57b54f1c705,2023-04-17,8642443,8642648,1312.0,1475.0,-163.0,1393.5,0,0,0.5,0.5,0.5,0.5,-1.0,-1.0,0,0.5,0
3,552b178c4e8d47c9feda614055c7e1c5,2023-04-17,8603901,8642451,2412.0,1515.0,897.0,1963.5,0,0,0.5,0.5,0.5,0.5,-1.0,-1.0,0,0.5,2
4,3d52f506bd0651ee6dc973b1fe2ef49e,2023-04-17,8633444,8603090,1610.0,2498.0,-888.0,2054.0,0,0,0.5,0.5,0.5,0.5,-1.0,-1.0,0,0.5,0


In [3]:
features.describe()

,date,elo_white,elo_black,elo_diff,elo_mean,white_games_count,black_games_count,white_form_10,black_form_10,white_form_30,black_form_30,days_since_last_game_white,days_since_last_game_black,h2h_games,h2h_white_score,y
count,471247,471267.000000,471267.000000,471267.000000,471267.000000,471267.000000,471267.000000,471267.000000,471267.000000,471267.000000,471267.000000,471247.000000,471247.000000,471267.000000,471267.000000,471267.000000
mean,2026-02-25 09:41:59.022932,2245.232675,2242.674263,2.558412,2243.953469,155.848300,154.844608,0.503778,0.504329,0.504647,0.504292,4.729931,4.827197,0.176189,0.503547,1.067607
min,2023-04-17 00:00:00,1083.000000,1160.000000,-1161.000000,1162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000
25%,2025-10-22 00:00:00,2088.000000,2085.000000,-154.000000,2114.500000,12.000000,12.000000,0.400000,0.400000,0.416667,0.416667,0.000000,0.000000,0.000000,0.500000,0.000000
50%,2026-03-01 00:00:00,2260.000000,2257.000000,3.000000,2254.000000,54.000000,54.000000,0.500000,0.500000,0.500000,0.500000,0.000000,0.000000,0.000000,0.500000,1.000000
75%,2026-07-10 00:00:00,2414.000000,2411.000000,160.000000,2391.000000,187.000000,185.000000,0.611111,0.611111,0.615385,0.611111,1.000000,1.000000,0.000000,0.500000,2.000000
max,2026-10-19 00:00:00,2841.000000,2841.000000,1191.000000,2827.500000,2363.000000,2360.000000,1.000000,1.000000,1.000000,1.000000,1193.000000,1195.000000,27.000000,1.000000,2.000000
std,NaN,243.896912,243.980173,227.203087,215.871961,246.658648,245.657544,0.186311,0.186543,0.167222,0.167343,27.123597,27.970656,0.792408,0.128871,0.891426


In [4]:
features[
    [
        "white_form_10",
        "black_form_10",
        "white_form_30",
        "black_form_30"
    ]
].describe()

,white_form_10,black_form_10,white_form_30,black_form_30
count,471267.000000,471267.000000,471267.000000,471267.000000
mean,0.503778,0.504329,0.504647,0.504292
std,0.186311,0.186543,0.167222,0.167343
min,0.000000,0.000000,0.000000,0.000000
25%,0.400000,0.400000,0.416667,0.416667
50%,0.500000,0.500000,0.500000,0.500000
75%,0.611111,0.611111,0.615385,0.611111
max,1.000000,1.000000,1.000000,1.000000


In [5]:
import duckdb

con = duckdb.connect(
    "../data/processed/chess.duckdb"
)

games = con.sql("""
SELECT
    white_id,
    black_id
FROM games_matched
WHERE white_id IS NOT NULL
AND black_id IS NOT NULL
""").df()

pairs = (
    games
    .groupby(
        ["white_id", "black_id"]
    )
    .size()
)

print("Unique pairings:", len(pairs))

print("\nPairings with >= 2 games:")
print((pairs >= 2).sum())

print("\nPairings with >= 5 games:")
print((pairs >= 5).sum())

print("\nPairings with >= 10 games:")
print((pairs >= 10).sum())

Unique pairings: 430200

Pairings with >= 2 games:
30939

Pairings with >= 5 games:
1584

Pairings with >= 10 games:
188


In [6]:
games.columns.tolist()

['white_id', 'black_id']

In [7]:
con = duckdb.connect("../data/processed/chess.duckdb")

games = con.sql("""
SELECT *
FROM games_matched
LIMIT 5
""").df()

print(games.columns.tolist())

['event', 'site', 'date_raw', 'round', 'white', 'black', 'white_elo', 'black_elo', 'white_fide_id', 'black_fide_id', 'white_title', 'black_title', 'event_date', 'ply_count', 'result', 'eco', 'date', 'date_approx', 'result_score', 'game_id', 'white_id', 'black_id']


In [8]:
import duckdb

con = duckdb.connect(
    "../data/processed/chess.duckdb"
)

eco_stats = con.sql("""
SELECT
    eco,
    COUNT(*) AS games,
    AVG(result_score / 2.0) AS white_score,
    AVG(
        CASE
            WHEN result = '1/2-1/2'
            THEN 1
            ELSE 0
        END
    ) AS draw_rate
FROM games_matched
GROUP BY eco
HAVING COUNT(*) >= 500
ORDER BY games DESC
""").df()

eco_stats.head(20)

,eco,games,white_score,draw_rate
0,A45,13191,0.522326,0.139337
1,A05,10236,0.563062,0.192165
2,D02,9775,0.551458,0.202864
3,B40,9210,0.534419,0.189142
4,B12,8612,0.535648,0.196935
5,B90,7754,0.518120,0.198994
6,A04,7482,0.554932,0.145683
7,A07,7437,0.556945,0.227511
8,B30,7116,0.533867,0.214165
9,B23,6893,0.507326,0.155810


In [9]:
eco_stats.describe()


,games,white_score,draw_rate
count,193.000000,193.000000,193.000000
mean,2246.383420,0.531981,0.206800
std,2210.156404,0.041189,0.058840
min,505.000000,0.277486,0.093722
25%,821.000000,0.507326,0.169794
50%,1256.000000,0.533867,0.200622
75%,2909.000000,0.559177,0.235211
max,13191.000000,0.617606,0.548780


In [10]:
eco_stats.sort_values(
    "white_score",
    ascending=False
).head(10)

,eco,games,white_score,draw_rate
157,E17,710,0.617606,0.235211
142,A35,832,0.611779,0.266827
117,D78,1094,0.603291,0.206581
68,D11,1902,0.600158,0.230810
123,D15,1066,0.598030,0.169794
48,B31,2909,0.597800,0.223444
115,A30,1104,0.596920,0.228261
95,C65,1259,0.596108,0.291501
130,C43,950,0.594211,0.325263
121,A11,1076,0.592937,0.219331


In [11]:
eco_stats.sort_values(
    "draw_rate",
    ascending=False
).head(10)

,eco,games,white_score,draw_rate
100,C67,1230,0.536179,0.548780
178,D41,580,0.591379,0.406897
124,C84,1048,0.582538,0.344466
66,C47,1991,0.506027,0.343044
130,C43,950,0.594211,0.325263
146,D27,779,0.578947,0.323492
45,C42,3084,0.577335,0.321984
63,D38,2084,0.561900,0.310940
160,E49,686,0.573615,0.310496
111,C88,1129,0.554473,0.299380


In [12]:
eco_stats[
    [
        "eco",
        "games",
        "white_score",
        "draw_rate"
    ]
].head()

,eco,games,white_score,draw_rate
0,A45,13191,0.522326,0.139337
1,A05,10236,0.563062,0.192165
2,D02,9775,0.551458,0.202864
3,B40,9210,0.534419,0.189142
4,B12,8612,0.535648,0.196935


In [ ]:
eco_stats.to_dict("records")[:5]


[{'eco': 'A45',
  'games': 13191,
  'white_score': 0.522325828216208,
  'draw_rate': 0.1393374270335835},
 {'eco': 'A05',
  'games': 10236,
  'white_score': 0.5630617428683079,
  'draw_rate': 0.19216490816725285},
 {'eco': 'D02',
  'games': 9775,
  'white_score': 0.5514578005115089,
  'draw_rate': 0.20286445012787724},
 {'eco': 'B40',
  'games': 9210,
  'white_score': 0.5344191096634093,
  'draw_rate': 0.18914223669923996},
 {'eco': 'B12',
  'games': 8612,
  'white_score': 0.5356479331165815,
  'draw_rate': 0.19693450998606596}]

: 